# 📓 Notebook 9 — Scikit-Learn Basics: Your First ML Models

> **Module:** Machine Learning · **Estimated time:** 60–75 min · **Difficulty:** Beginner / Intermediate

This is where everything you have learned so far converges. NumPy gave us arrays, pandas gave us tables, matplotlib gave us charts. Now scikit-learn gives us a uniform, beautifully designed toolkit for **training, evaluating, and using machine-learning models**.

We will build two end-to-end ML pipelines:

1. **Classification** — predicting the species of an iris flower from its measurements.
2. **Regression** — predicting California house prices from neighbourhood features.

## 🎯 Learning objectives

By the end of this notebook you will:

1. Recognise the **classification vs regression** distinction.
2. Follow the **scikit-learn API contract**: `fit`, `predict`, `score`.
3. Use **`train_test_split`** correctly and explain why.
4. Train and compare multiple models (logistic regression, decision tree, random forest, k-NN, linear regression).
5. Evaluate models with **accuracy, precision, recall, confusion matrix** (classification) and **MSE, MAE, R²** (regression).
6. Use a **Pipeline** to combine preprocessing and modelling.
7. Tune hyperparameters with **cross-validation**.
8. Interpret models via **feature importance** and prediction inspection.

## ✅ Prerequisites

Notebooks 1–8.

## 1. The ML workflow in one picture

```
                     ┌──────────────────────────────────────────┐
                     │  1. Load & explore                       │
                     │  2. Prepare features (X) and target (y)  │
   Raw data ────────►│  3. Split into train / test              │
                     │  4. Fit  model on train  (`.fit`)        │
                     │  5. Predict on test     (`.predict`)     │
                     │  6. Evaluate vs y_test  (`.score`, …)    │
                     │  7. Iterate, tune, deploy                │
                     └──────────────────────────────────────────┘
```

**Two unbreakable principles:**

- **Never train on data you evaluate on.** The whole point of "test data" is to estimate how the model behaves on *unseen* examples.
- **Look at the data first.** A 10-minute glance saves hours of debugging.

## 2. Setup

In [ ]:
# The "Big 4" + scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Datasets
from sklearn.datasets import load_iris, fetch_california_housing

# Splitting & validation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Pipelines
from sklearn.pipeline import Pipeline

# Models — classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Models — regression
from sklearn.linear_model import LinearRegression

# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
)

# Reproducibility
RANDOM_STATE = 42

# Friendlier plot defaults (matches NB 8)
plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")

## 3. Project 1 — Iris classification 🌸

The **Iris** dataset is the "Hello World" of ML. 150 iris flowers, each described by 4 numeric measurements (sepal length, sepal width, petal length, petal width). Goal: predict the **species** (setosa, versicolor, virginica) from the measurements.

### 3.1 Load and explore

In [ ]:
iris = load_iris()

# Move into a DataFrame so we can use pandas/seaborn-style operations
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Shape: {iris_df.shape}")
print(f"Classes: {iris.target_names.tolist()}")
iris_df.head()

In [ ]:
iris_df.describe()

In [ ]:
# Per-class counts — make sure it is balanced
print(iris_df["species"].value_counts())

### 3.2 Visual exploration

A small `pairplot` is the single best first look at a tabular dataset. We will use matplotlib only (so we don\'t depend on seaborn for this).

In [ ]:
features = iris.feature_names
fig, axes = plt.subplots(len(features), len(features), figsize=(11, 11))
fig.suptitle("Iris pair-plot (diagonal = histogram per class)", fontsize=14)

palette = ["#4C72B0", "#DD8452", "#55A467"]

for i in range(len(features)):
    for j in range(len(features)):
        ax = axes[i, j]
        if i == j:
            # Diagonal: histogram per class
            for k, cls in enumerate(iris.target_names):
                ax.hist(iris.data[iris.target == k, i], bins=12,
                        alpha=0.55, color=palette[k], label=cls)
            if i == 0:
                ax.legend(fontsize=7)
        else:
            for k, cls in enumerate(iris.target_names):
                ax.scatter(iris.data[iris.target == k, j],
                           iris.data[iris.target == k, i],
                           s=12, alpha=0.7, color=palette[k])
        if i == len(features) - 1:
            ax.set_xlabel(features[j], fontsize=8)
        if j == 0:
            ax.set_ylabel(features[i], fontsize=8)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

**Reading the pair-plot.** *setosa* (blue) is linearly separable from the other two species — you can draw a straight line in any panel that separates blue from orange+green. *versicolor* and *virginica* overlap a bit on most features but are well separated on **petal length / width**. This already tells us our classifier should reach ~95 %+ accuracy.

### 3.3 Train / test split

We hold out **20 %** of the data as a test set we won\'t touch until the very end. `stratify=y` makes sure the class proportions stay the same in both splits — important when classes are imbalanced.

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train : {X_train.shape}  (y distribution: {np.bincount(y_train)})")
print(f"Test  : {X_test.shape}   (y distribution: {np.bincount(y_test)})")

### 3.4 The scikit-learn API contract

Every estimator follows the same shape — that\'s the genius of scikit-learn:

```python
model = SomeModel(hyperparam=value)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
score  = model.score(X_test, y_test)
```

This means swapping a logistic regression for a random forest is a *one-line* change.

In [ ]:
# Train four different classifiers and compare them on the test set
classifiers = {
    "Logistic Regression":  LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
    "Decision Tree":        DecisionTreeClassifier(random_state=RANDOM_STATE),
    "k-NN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

print(f"{'Model':<20}{'Accuracy':>10}")
print("-" * 30)
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    print(f"{name:<20}{acc:>10.3f}")

All four models score very high — this dataset is *easy*. The interesting practice question is not "which is best?" but "how big is the typical *spread* between models?" — here it is tiny, which is a good sign that the signal is strong.

### 3.5 A closer look — the confusion matrix

Accuracy hides important detail. The **confusion matrix** shows where the model is right and where it is wrong, broken down by predicted vs true class.

In [ ]:
rf = classifiers["Random Forest"]
y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), iris.target_names)
ax.set_yticks(range(3), iris.target_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix — Random Forest")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

**Reading the matrix.** Diagonal = correct predictions, off-diagonal = mistakes. The `classification_report` shows three additional metrics per class:

- **Precision** = "of the times the model said *X*, how often was it right?"
- **Recall** = "of the actual *X*s, how many did the model catch?"
- **F1-score** = harmonic mean of precision and recall, a single summary number.

Different applications care about different metrics — a spam filter prioritises *precision* (don\'t flag legit mail), a medical screening test prioritises *recall* (don\'t miss anyone sick).

### 3.6 Predicting new flowers

In [ ]:
# Three hand-crafted measurements: [sepal_len, sepal_wid, petal_len, petal_wid]
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],   # looks like setosa
    [6.2, 2.8, 4.8, 1.8],   # could be either versicolor or virginica
    [7.7, 3.0, 6.1, 2.3],   # likely virginica
])

pred = rf.predict(new_flowers)
prob = rf.predict_proba(new_flowers)

for i, p in enumerate(pred):
    probs = ", ".join(f"{iris.target_names[k]}: {prob[i, k]:.0%}" for k in range(3))
    print(f"Flower {i+1}: predicted {iris.target_names[p]:<12}   ({probs})")

`predict_proba` returns the model\'s probability per class. Look at flower 2 — even the model is uncertain whether it is versicolor or virginica. That uncertainty is information you should expose to users.

### 3.7 Feature importance

A trained Random Forest can tell you *which features it relied on most*. This is a great quick-and-dirty interpretability tool.

In [ ]:
importance = rf.feature_importances_
order = np.argsort(importance)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(np.array(iris.feature_names)[order], importance[order],
        color="#4C72B0", edgecolor="black")
ax.set_title("Random-Forest feature importance — Iris")
ax.set_xlabel("Relative importance")
plt.tight_layout()
plt.show()

for f, imp in zip(np.array(iris.feature_names)[order[::-1]], np.sort(importance)[::-1]):
    print(f"  {f:<22}: {imp:.3f}")

Petal width and petal length dominate — confirming the visual intuition from the pair-plot.

## 4. Project 2 — California housing 🏠

Now for a **regression** task: predict the *median house value* in each California census block from neighbourhood features. We use `fetch_california_housing` because the old Boston dataset has been removed from scikit-learn for ethical reasons.

In [ ]:
cal = fetch_california_housing()

cal_df = pd.DataFrame(cal.data, columns=cal.feature_names)
cal_df["price"] = cal.target   # in $100 000s

print(f"Shape  : {cal_df.shape}")
print(f"Target : median house value (in $100,000)")
cal_df.head()

In [ ]:
cal_df.describe().round(2)

### 4.1 Visual sanity check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Target distribution
axes[0].hist(cal_df["price"], bins=40, color="#4C72B0", edgecolor="black")
axes[0].set_title("Median house value")
axes[0].set_xlabel("$100,000s")
axes[0].set_ylabel("Count")

# Median income vs price — should be strong
axes[1].scatter(cal_df["MedInc"], cal_df["price"], alpha=0.2, s=8, color="#4C72B0")
axes[1].set_title("Median income vs price")
axes[1].set_xlabel("Median income")
axes[1].set_ylabel("Price ($100k)")

# Geo-spatial peek (rough!): lon/lat coloured by price
sc = axes[2].scatter(cal_df["Longitude"], cal_df["Latitude"],
                     c=cal_df["price"], cmap="viridis", alpha=0.3, s=4)
axes[2].set_title("Spatial distribution")
axes[2].set_xlabel("Longitude")
axes[2].set_ylabel("Latitude")
fig.colorbar(sc, ax=axes[2], label="Price")

plt.tight_layout()
plt.show()

**Observations.**

- Prices are clipped at the top (note the spike near 5 — that\'s an artefact of the dataset, not nature).
- *Median income* and *price* are clearly positively correlated.
- The spatial map roughly outlines California: high-price clusters around the Bay Area and Los Angeles.

### 4.2 Train / test split and baseline

In [ ]:
X = cal.data
y = cal.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape},  Test: {X_test.shape}")

### 4.3 A first pipeline — scaler + linear regression

Many models care about feature scales. Median income is on a scale of 0–15; total population is in the thousands. We standardise inside a **Pipeline** so the scaler is fit on the training data only — never the test data. (That is the leak that ruins many beginner projects.)

In [ ]:
pipe_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LinearRegression()),
])

pipe_linear.fit(X_train, y_train)
y_pred = pipe_linear.predict(X_test)

print(f"Linear Regression:")
print(f"  MAE  = {mean_absolute_error(y_test, y_pred):.3f}")
print(f"  RMSE = {mean_squared_error(y_test, y_pred) ** 0.5:.3f}")
print(f"  R²   = {r2_score(y_test, y_pred):.3f}")

**How to read regression metrics:**

- **MAE** (mean absolute error): on average the prediction is off by this much (in target units; here in $100k).
- **RMSE** (root mean squared error): penalises large errors more — useful when occasional huge errors matter.
- **R²** (coefficient of determination): proportion of variance explained, in [-∞, 1]. 1.0 = perfect, 0 = same as predicting the mean.

### 4.4 A stronger model — random forest

Random forests fit non-linear patterns and interactions out of the box.

In [ ]:
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),       # not strictly needed for trees, but harmless
    ("model",  RandomForestRegressor(n_estimators=80, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)),
])

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print(f"Random Forest:")
print(f"  MAE  = {mean_absolute_error(y_test, y_pred_rf):.3f}")
print(f"  RMSE = {mean_squared_error(y_test, y_pred_rf) ** 0.5:.3f}")
print(f"  R²   = {r2_score(y_test, y_pred_rf):.3f}")

The random forest typically beats linear regression by a lot here — confirming that the relationship is non-linear.

### 4.5 Predicted vs actual — a diagnostic plot

The single best regression-debugging chart: plot predicted values against true ones. A perfect model lies on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, y_hat, name in [(axes[0], y_pred,    "Linear Regression"),
                         (axes[1], y_pred_rf, "Random Forest")]:
    ax.scatter(y_test, y_hat, alpha=0.25, s=8)
    lim = [min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())]
    ax.plot(lim, lim, "r--", linewidth=1)
    ax.set_xlabel("True value (price)")
    ax.set_ylabel("Predicted value")
    ax.set_title(f"{name}\nR² = {r2_score(y_test, y_hat):.3f}")

plt.tight_layout()
plt.show()

### 4.6 Cross-validation — a more honest score

A single train/test split is one *sample* of how the model behaves. Cross-validation runs `k` splits and averages — much more reliable.

In [ ]:
scores = cross_val_score(pipe_rf, X, y, cv=3, scoring="r2", n_jobs=-1)
print(f"3-fold R² scores: {scores.round(3)}")
print(f"Mean ± std       : {scores.mean():.3f} ± {scores.std():.3f}")

### 4.7 Hyperparameter tuning with `GridSearchCV`

A typical question: "is 200 trees better than 100? Should the trees be deeper?". `GridSearchCV` automates the experiment.

In [ ]:
param_grid = {
    "model__n_estimators": [80, 150],
    "model__max_depth":    [15, None],
}

# Small grid + cv=2 keeps this fast in the notebook. On a real project bump cv to 5+
grid = GridSearchCV(pipe_rf, param_grid, cv=2, n_jobs=-1, scoring="r2")
grid.fit(X_train, y_train)

print(f"Best params : {grid.best_params_}")
print(f"Best CV R²  : {grid.best_score_:.3f}")
print(f"Test  R²    : {grid.score(X_test, y_test):.3f}")

### 4.8 Which features matter?

In [ ]:
rf_model = pipe_rf.named_steps["model"]
importance = rf_model.feature_importances_
order = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(cal.feature_names)[order], importance[order],
        color="#4C72B0", edgecolor="black")
ax.set_title("Random-Forest feature importance — California housing")
ax.set_xlabel("Relative importance")
plt.tight_layout()
plt.show()

> 💡 The biggest predictor of house price is the **median income** of the block — unsurprising! Latitude and longitude come next, which explains why the spatial map looked so informative earlier.

## 5. Common ML pitfalls (and how to avoid them)

| Pitfall                                                | Symptom                                       | Fix                                                    |
|--------------------------------------------------------|-----------------------------------------------|--------------------------------------------------------|
| **Data leakage** through preprocessing                  | suspiciously good test score                  | Put preprocessing inside a `Pipeline`                  |
| Training on the test set                               | unrealistically high accuracy                 | Never call `.fit()` on data you will evaluate on        |
| Ignoring class imbalance                               | high accuracy but useless predictions         | Use `stratify=y`, `class_weight`, look at per-class metrics |
| Tuning hyperparameters on the test set                  | optimistic generalisation estimate            | Use cross-validation or a separate validation set      |
| Forgetting to scale features                           | weak performance for distance-based models    | Add `StandardScaler` in your pipeline                  |
| Reporting only accuracy                                | misleading on imbalanced data                  | Always also report precision / recall / F1 / confusion |
| Tree-based models on extrapolation                     | wildly wrong predictions outside training range | Use linear models or be honest about limits            |


## 🧪 Practice exercises

### Exercise 1 — A different classifier on Iris

Train a **k-nearest-neighbours classifier with `k=3`** on Iris (same train/test split). Compute accuracy, classification report and confusion matrix.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           stratify=y, random_state=42)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_tr, y_tr)
print(f"Accuracy: {knn.score(X_te, y_te):.3f}")

print(classification_report(y_te, knn.predict(X_te), target_names=iris.target_names))
print(confusion_matrix(y_te, knn.predict(X_te)))
```
</details>

### Exercise 2 — Wine quality (multi-class)

Load the wine dataset (`from sklearn.datasets import load_wine`). Train a random-forest classifier, report the cross-validated accuracy, and plot the feature importances.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.datasets import load_wine

wine = load_wine()
X, y = wine.data, wine.target

clf = RandomForestClassifier(n_estimators=200, random_state=42)
scores = cross_val_score(clf, X, y, cv=5)
print(f"CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

clf.fit(X, y)
order = np.argsort(clf.feature_importances_)
plt.figure(figsize=(8, 6))
plt.barh(np.array(wine.feature_names)[order], clf.feature_importances_[order])
plt.title("Feature importance — wine classifier")
plt.tight_layout()
plt.show()
```
</details>

### Exercise 3 — Regression with a feature subset

On California housing, train a `LinearRegression` using **only `MedInc` and `AveRooms`**. Report R² and compare with the full-feature model.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
cal = fetch_california_housing()
X = cal.data[:, [0, 2]]              # MedInc, AveRooms
y = cal.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
pipe = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())])
pipe.fit(X_tr, y_tr)

print(f"R² with 2 features  : {pipe.score(X_te, y_te):.3f}")
# Compare with the full model from earlier (~0.61 for linear, ~0.81 for RF).
```

We expect a much lower R² — most of the signal lives in the omitted features (especially the geo-coordinates).
</details>

### Exercise 4 — Tune k for k-NN

On Iris, scan `k ∈ {1, 3, 5, 7, 9, 15, 25}` with 5-fold cross-validation and plot mean accuracy vs k.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
iris = load_iris()
ks = [1, 3, 5, 7, 9, 15, 25]
means = []
for k in ks:
    s = cross_val_score(KNeighborsClassifier(n_neighbors=k), iris.data, iris.target, cv=5)
    means.append(s.mean())

plt.figure(figsize=(7, 4))
plt.plot(ks, means, marker="o")
plt.title("k-NN accuracy vs k on Iris")
plt.xlabel("k")
plt.ylabel("CV accuracy")
plt.tight_layout()
plt.show()
```

You should see a peak around `k = 5–9`, with small `k` overfitting and very large `k` underfitting.
</details>

### Exercise 5 — Debug me 🐞

The cell below is supposed to evaluate a logistic-regression classifier on Iris. The reported accuracy is **suspiciously perfect**. What\'s wrong? Fix it.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

model = LogisticRegression(max_iter=200)
model.fit(X, y)
acc = model.score(X, y)
print(f"Accuracy: {acc:.3f}")


<details>
<summary>💡 <b>Solution</b></summary>

The model is evaluated on the **same data it was trained on** → no test set. Fix:

```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           stratify=y, random_state=42)
model = LogisticRegression(max_iter=200).fit(X_tr, y_tr)
print(f"Test accuracy: {model.score(X_te, y_te):.3f}")
```

The number you print on the *training* set tells you almost nothing — a sufficiently flexible model can memorise the training data perfectly. Generalisation is what matters.
</details>

## 🎁 Bonus mini-project — Your own ML cycle

Pick **either** the wine or the breast-cancer dataset (`from sklearn.datasets import load_breast_cancer`). Then:

1. Explore the data (shape, class balance, a couple of plots).
2. Split train/test 80/20 with stratification.
3. Train **three** different models inside `Pipeline`s with `StandardScaler`.
4. Compare them with cross-validation.
5. Pick the best and tune one hyperparameter with `GridSearchCV`.
6. Report final test-set accuracy, classification report, and confusion matrix.

## 🧠 Key takeaways

1. Every scikit-learn estimator follows the same `fit / predict / score` contract — interchangeable building blocks.
2. **Always** split into train/test (and prefer cross-validation). Never train on your test set.
3. Use a **Pipeline** to bundle preprocessing with the model. This is how you avoid data leakage.
4. Pick metrics that match your problem: accuracy/F1 for classification; MAE/RMSE/R² for regression.
5. Don\'t stop at "the score" — look at the **confusion matrix**, **predicted-vs-actual plot**, and **feature importance**.
6. Cross-validation gives you a much more honest score than a single hold-out.
7. `GridSearchCV` is the simplest hyperparameter-tuning tool — start there before reaching for more advanced techniques.

## ✅ Self-assessment

- [ ] Split data with `train_test_split(stratify=y)`.
- [ ] Wrap preprocessing + model in a `Pipeline`.
- [ ] Compute accuracy, confusion matrix, classification report.
- [ ] Compute MAE, RMSE, R² for a regression model.
- [ ] Run a small `GridSearchCV` to tune one hyperparameter.
- [ ] Inspect and interpret `feature_importances_` from a Random Forest.

## 🚀 Next step

You are ready for the **capstone (Notebook 10)** — a multi-step end-to-end analysis of weather data that uses everything you have learned.